# CMPE 272 In-Class: Datastore Speed Dating (SpartanEats)

**Session 9, datastores, databases and query languages.**
**Group assignment: one notebook per project team, in your Canvas project team.**
**Section 49: due tonight, Wednesday, October 14, 11:59 PM. Section 03: due Tuesday, October 20, 11:59 PM.**

SpartanEats is a food-delivery app in the South Bay. It keeps the same data in six stores, one of each
family from tonight's lecture: **relational, document, key-value, wide-column, graph and vector.**
Your team decides which store should answer each of eight app requests, then tests the choice by
writing the query and measuring what it costs.

Every store counts the **items it touches**: rows read, keys read, documents read, partition rows read,
edges followed, vectors compared. One unit for all six stores, so you can compare them. Fewer items
touched means less work for the database.

## How to work on it

1. Go to <https://colab.research.google.com>, choose **File > Upload notebook**, and pick this file.
2. Choose **File > Save a copy in Drive before you type anything.** Edits to the uploaded file are lost when the tab closes.
3. Share your Drive copy with your teammates, so the whole team works in one notebook.
4. Run the setup cell. It builds SpartanEats' data and the six stores in a few seconds.
5. Fill in every `REPLACE`, round by round. In round 2, each query cell checks itself and prints `PASS` or `NOT YET`.
6. Run the **validation cell** at the end. It lists anything that is still missing.
7. Choose **Runtime > Restart and run all**, and confirm that every cell has output.
8. Choose **File > Download > Download .ipynb**. One team member uploads that file to the Canvas assignment **In-class exercise: Data Model**.

## Rounds and time (about 40 minutes)

| Round | What you do | Time |
|---|---|---|
| 1. Match | For each of the eight requests P1 to P8, pick a store family, the access key and a reason | 10 min |
| 2. Date | Write the query for P2, P3, P4 and P5 against the store you picked. The notebook checks the answer and shows its cost next to a SQL baseline. P6 is optional | 20 min |
| 3. Two-store budget | The CTO pays for only two stores. Reassign all eight requests and say what you lose | 10 min |

The notebook is graded on completeness. The cost score at the end is for comparison only.
If something will not work, write `BROKE:` and what went wrong where the notebook offers it.
That is an honest answer and it counts as complete. A result you did not get does not.

In [1]:
# Every member of your team, with each name exactly as it appears in Canvas.
GROUP = [
    "Arda Korkmaz",
    "Mike Chau",
    "Meron Tesfandrias",
    "Mann Malviya"
    # Add or delete lines so the list matches your team.
]
TEAM_NAME = "Capybara Labs"

print("Team:", TEAM_NAME, "|", len(GROUP), "members")

Team: Capybara Labs | 4 members


## Setup: SpartanEats and its six stores

Run the next cell once. It generates the data from a fixed seed, so every team has the same data,
and loads it into the six stores. Nothing to fill in. You do not need to read the code; the
**store guide** after it lists everything you can call.

In [2]:
# SpartanEats data and stores. Nothing to change in this cell.
import copy, datetime, hashlib, json, math, random, re, sqlite3
import numpy as np

FAMILIES = ["relational", "document", "key-value", "wide-column", "graph", "vector"]
SEED = 272
CITIES = ["San Jose", "Santa Clara", "Sunnyvale", "Milpitas", "Campbell"]
CITY_W = [40, 20, 18, 12, 10]
TAGS = ["spicy", "vegetarian", "noodles", "rice", "grilled", "fried", "soup", "sweet"]
MENUS = {
    "Mexican": (["Taqueria", "Cantina", "Cocina"], [("Carne Asada Tacos", ["grilled", "spicy"]), ("Bean Burrito", ["vegetarian", "rice"]), ("Pozole", ["soup", "spicy"]), ("Churros", ["sweet", "fried"]), ("Chicken Enchiladas", ["spicy"]), ("Veggie Quesadilla", ["vegetarian", "grilled"])]),
    "Vietnamese": (["Pho House", "Banh Mi Shop", "Kitchen"], [("Pho Tai", ["noodles", "soup"]), ("Banh Mi", ["grilled"]), ("Bun Bo Hue", ["noodles", "soup", "spicy"]), ("Com Tam", ["rice", "grilled"]), ("Tofu Spring Rolls", ["vegetarian", "fried"]), ("Che Ba Mau", ["sweet"])]),
    "Indian": (["Curry House", "Tandoor", "Dhaba"], [("Chicken Tikka Masala", ["spicy", "rice"]), ("Chana Masala", ["vegetarian", "spicy"]), ("Tandoori Chicken", ["grilled", "spicy"]), ("Vegetable Biryani", ["rice", "vegetarian"]), ("Samosa", ["fried", "vegetarian"]), ("Gulab Jamun", ["sweet", "fried"])]),
    "Italian": (["Trattoria", "Pizzeria", "Osteria"], [("Margherita Pizza", ["vegetarian"]), ("Spaghetti Carbonara", ["noodles"]), ("Minestrone", ["soup", "vegetarian"]), ("Chicken Parmesan", ["fried"]), ("Mushroom Risotto", ["rice", "vegetarian"]), ("Tiramisu", ["sweet"])]),
    "Chinese": (["Noodle Bar", "Dumpling House", "Wok"], [("Beef Chow Fun", ["noodles"]), ("Mapo Tofu", ["spicy", "vegetarian"]), ("Pork Dumplings", ["fried"]), ("Hot and Sour Soup", ["soup", "spicy"]), ("Yangzhou Fried Rice", ["rice", "fried"]), ("Sesame Balls", ["sweet", "fried"])]),
    "Japanese": (["Ramen Ya", "Sushi Bar", "Izakaya"], [("Tonkotsu Ramen", ["noodles", "soup"]), ("Salmon Nigiri", ["rice"]), ("Chicken Katsu", ["fried", "rice"]), ("Vegetable Tempura", ["fried", "vegetarian"]), ("Yakitori", ["grilled"]), ("Mochi Ice Cream", ["sweet"])]),
    "Thai": (["Thai Kitchen", "Thai Bistro", "Street Thai"], [("Pad Thai", ["noodles"]), ("Green Curry", ["spicy", "rice"]), ("Tom Yum Soup", ["soup", "spicy"]), ("Basil Fried Rice", ["rice", "spicy"]), ("Tofu Satay", ["grilled", "vegetarian"]), ("Mango Sticky Rice", ["sweet", "rice"])]),
    "American": (["Grill", "Burger Joint", "Diner"], [("Cheeseburger", ["grilled"]), ("Fried Chicken Sandwich", ["fried"]), ("BBQ Ribs", ["grilled"]), ("Mac and Cheese", ["vegetarian"]), ("Clam Chowder", ["soup"]), ("Apple Pie", ["sweet"])]),
}
CUISINES = list(MENUS)
ADJ = ["Golden", "Blue", "Lucky", "Little", "Happy", "Red", "Green", "Silver", "Sunny", "Old Town",
       "Spartan", "Downtown", "Garden", "Corner", "Royal"]
FIRST = ["Ana", "Bo", "Chris", "Dana", "Eli", "Fatima", "Gabe", "Hana", "Ivan", "Jia", "Kai", "Lena", "Mo",
         "Nia", "Omar", "Priya", "Quinn", "Rosa", "Sam", "Tara", "Uma", "Vic", "Wen", "Xavi", "Yuki", "Zoe"]
LAST = ["Nguyen", "Garcia", "Patel", "Chen", "Kim", "Lopez", "Singh", "Tran", "Wong", "Smith", "Reyes",
        "Khan", "Park", "Silva", "Ali", "Ramos"]
DAY0 = datetime.date(2026, 9, 14)          # 30 days of orders: Sep 14 to Oct 13, 2026
HOUR_W = [1, 1, 2, 3, 6, 8, 6, 3, 2, 3, 6, 8, 8, 6, 3, 1]   # hours 7 to 22


def generate(seed=SEED):
    rng = random.Random(seed)
    customers = [{"customer_id": f"C-{i:04d}", "name": f"{rng.choice(FIRST)} {rng.choice(LAST)}",
                  "city": rng.choices(CITIES, CITY_W)[0]} for i in range(1, 2001)]
    ranks = list(range(1, 150))
    rng.shuffle(ranks)
    restaurants, dishes = [], []
    for i in range(1, 151):
        cuisine = CUISINES[(i * 5 + rng.randrange(3)) % len(CUISINES)]
        words, menu = MENUS[cuisine]
        rank = 0 if i == 7 else ranks.pop()
        r = {"restaurant_id": f"R-{i:03d}", "name": f"{rng.choice(ADJ)} {rng.choice(words)}",
             "cuisine": cuisine, "city": rng.choices(CITIES, CITY_W)[0], "pop": 1.0 / (1 + rank) ** 0.8}
        restaurants.append(r)
        for name, tags in rng.sample(menu, 4):
            v = [0.0] * 16
            v[CUISINES.index(cuisine)] = 1.0
            for t in tags:
                v[8 + TAGS.index(t)] = 0.8
            v = [x + rng.gauss(0, 0.3) for x in v]
            n = math.sqrt(sum(x * x for x in v))
            dishes.append({"dish_id": f"D-{len(dishes) + 1:04d}", "restaurant_id": r["restaurant_id"],
                           "name": name, "cuisine": cuisine, "tags": list(tags),
                           "price_cents": rng.randrange(899, 2399, 50),
                           "embedding": [round(x / n, 6) for x in v]})
    menu_of = {}
    for d in dishes:
        menu_of.setdefault(d["restaurant_id"], []).append(d)
    by_city = {c: [r for r in restaurants if r["city"] == c] for c in CITIES}
    activity = [rng.lognormvariate(0, 0.8) for _ in customers]
    buyers = rng.choices(customers, activity, k=20000)
    raw = []
    for c in buyers:
        pool = by_city[c["city"]] if rng.random() < 0.7 and by_city[c["city"]] else restaurants
        r = rng.choices(pool, [x["pop"] for x in pool])[0]
        minute = rng.randrange(30) * 1440 + (7 + rng.choices(range(16), HOUR_W)[0]) * 60 + rng.randrange(60)
        lines = []
        for d in rng.sample(menu_of[r["restaurant_id"]], rng.choice([1, 1, 2, 2, 2, 3, 3, 4])):
            lines.append({"dish_id": d["dish_id"], "name": d["name"], "qty": rng.choice([1, 1, 1, 2, 2, 3]),
                          "price_cents": d["price_cents"]})
        raw.append((minute, c, r, lines))
    raw.sort(key=lambda t: t[0])
    orders = []
    for k, (minute, c, r, lines) in enumerate(raw, 1):
        t = datetime.datetime.combine(DAY0, datetime.time()) + datetime.timedelta(minutes=minute)
        orders.append({"order_id": f"O-{k:05d}", "customer_id": c["customer_id"], "customer_city": c["city"],
                       "restaurant_id": r["restaurant_id"], "restaurant_name": r["name"], "cuisine": r["cuisine"],
                       "city": r["city"], "ts": t.strftime("%Y-%m-%d %H:%M"), "day": t.strftime("%Y-%m-%d"),
                       "items": lines, "total_cents": sum(x["qty"] * x["price_cents"] for x in lines)})
    friends = set()
    city_members = {ct: [x["customer_id"] for x in customers if x["city"] == ct] for ct in CITIES}
    ids = [x["customer_id"] for x in customers]
    for c in customers:
        for _ in range(rng.choice([1, 2, 2, 3])):
            other = rng.choice(city_members[c["city"]]) if rng.random() < 0.7 else rng.choice(ids)
            if other != c["customer_id"]:
                friends.add(tuple(sorted((c["customer_id"], other))))
    carts = {}
    for i in range(1, 3001):
        c = rng.choice(customers)
        r = rng.choice(restaurants)
        carts[f"S-{i:05d}"] = {"customer_id": c["customer_id"], "restaurant_id": r["restaurant_id"],
                               "items": [{"dish_id": d["dish_id"], "qty": rng.choice([1, 2])}
                                         for d in rng.sample(menu_of[r["restaurant_id"]], rng.choice([1, 2, 3]))],
                               "updated": f"2026-10-14 {rng.randrange(8, 22):02d}:{rng.randrange(60):02d}"}
    for r in restaurants:
        del r["pop"]
    return {"customers": customers, "restaurants": restaurants, "dishes": dishes, "orders": orders,
            "friends": sorted(friends), "carts": carts}


class _Store:
    family = ""

    def __init__(self):
        self.touched = 0

    def reset(self):
        self.touched = 0


class RelationalStore(_Store):
    """SQLite in memory. Tables: customers, restaurants, dishes, orders, order_items, friends, carts,
    wallets, payouts. sql() counts every row SQLite reads, including index lookups in joins."""
    family = "relational"
    SCHEMA = """
    CREATE TABLE _customers  (customer_id TEXT PRIMARY KEY, name TEXT, city TEXT);
    CREATE TABLE _restaurants(restaurant_id TEXT PRIMARY KEY, name TEXT, cuisine TEXT, city TEXT);
    CREATE TABLE _dishes     (dish_id TEXT PRIMARY KEY, restaurant_id TEXT, name TEXT, cuisine TEXT,
                              tags TEXT, price_cents INTEGER, embedding TEXT);
    CREATE TABLE _orders     (order_id TEXT PRIMARY KEY, customer_id TEXT, restaurant_id TEXT,
                              ts TEXT, day TEXT, total_cents INTEGER);
    CREATE TABLE _order_items(order_id TEXT, dish_id TEXT, qty INTEGER, price_cents INTEGER);
    CREATE TABLE _friends    (a TEXT, b TEXT);
    CREATE TABLE _carts      (session_id TEXT PRIMARY KEY, customer_id TEXT, restaurant_id TEXT,
                              items TEXT, updated TEXT);
    CREATE TABLE _wallets    (customer_id TEXT PRIMARY KEY, balance_cents INTEGER);
    CREATE TABLE _payouts    (restaurant_id TEXT PRIMARY KEY, balance_cents INTEGER);
    CREATE INDEX restaurants_city ON _restaurants(city);
    CREATE INDEX dishes_restaurant ON _dishes(restaurant_id);
    CREATE INDEX orders_customer_ts ON _orders(customer_id, ts);
    CREATE INDEX orders_restaurant_day ON _orders(restaurant_id, day);
    CREATE INDEX orders_day ON _orders(day);
    CREATE INDEX order_items_order ON _order_items(order_id);
    CREATE INDEX friends_a ON _friends(a, b);
    """
    TABLES = ["customers", "restaurants", "dishes", "orders", "order_items", "friends", "carts", "wallets", "payouts"]

    def __init__(self, data):
        super().__init__()
        self.db = sqlite3.connect(":memory:")
        self.db.create_function("_touch", 2, self._count)
        self.db.executescript(self.SCHEMA)
        x = self.db.executemany
        x("INSERT INTO _customers VALUES (?,?,?)", [(c["customer_id"], c["name"], c["city"]) for c in data["customers"]])
        x("INSERT INTO _restaurants VALUES (?,?,?,?)", [(r["restaurant_id"], r["name"], r["cuisine"], r["city"]) for r in data["restaurants"]])
        x("INSERT INTO _dishes VALUES (?,?,?,?,?,?,?)", [(d["dish_id"], d["restaurant_id"], d["name"], d["cuisine"], ",".join(d["tags"]), d["price_cents"], json.dumps(d["embedding"])) for d in data["dishes"]])
        x("INSERT INTO _orders VALUES (?,?,?,?,?,?)", [(o["order_id"], o["customer_id"], o["restaurant_id"], o["ts"], o["day"], o["total_cents"]) for o in data["orders"]])
        x("INSERT INTO _order_items VALUES (?,?,?,?)", [(o["order_id"], i["dish_id"], i["qty"], i["price_cents"]) for o in data["orders"] for i in o["items"]])
        x("INSERT INTO _friends VALUES (?,?)", [p for a, b in data["friends"] for p in ((a, b), (b, a))])
        x("INSERT INTO _carts VALUES (?,?,?,?,?)", [(k, v["customer_id"], v["restaurant_id"], json.dumps(v["items"]), v["updated"]) for k, v in data["carts"].items()])
        x("INSERT INTO _wallets VALUES (?,?)", [(c["customer_id"], 2000) for c in data["customers"]])
        x("INSERT INTO _payouts VALUES (?,?)", [(r["restaurant_id"], 50000) for r in data["restaurants"]])
        for t in self.TABLES:
            self.db.execute(f"CREATE VIEW {t} AS SELECT * FROM _{t} WHERE _touch('{t}', rowid)")
        self.db.commit()

    def _count(self, table, rowid):
        self.touched += 1
        return 1

    def sql(self, query, params=()):
        """Run a SELECT and return its rows as a list of tuples."""
        return self.db.execute(query, params).fetchall()

    def explain(self, query, params=()):
        """Print SQLite's plan: SEARCH uses an index, SCAN reads the whole table.
        _orders is the table behind the orders view, and so on."""
        for row in self.db.execute("EXPLAIN QUERY PLAN " + query, params).fetchall():
            print("  ", row[-1])

    def schema(self):
        print(self.SCHEMA.replace("CREATE TABLE _", "CREATE TABLE ").replace(" ON _", " ON "))


class DocumentStore(_Store):
    """Orders as nested documents, keyed by order_id. A document carries the restaurant's name,
    cuisine and city and every line item, so one read renders an order screen."""
    family = "document"

    def __init__(self, data):
        super().__init__()
        self._docs = {o["order_id"]: o for o in data["orders"]}

    def get(self, order_id):
        """One document by its key. 1 item touched."""
        self.touched += 1
        d = self._docs.get(order_id)
        return copy.deepcopy(d) if d is not None else None

    def find(self, **equals):
        """Every document whose top-level fields equal the arguments, e.g. find(customer_id="C-0001").
        There is no secondary index, so this reads every document: 20,000 items touched."""
        self.touched += len(self._docs)
        return [copy.deepcopy(d) for d in self._docs.values() if all(d.get(k) == v for k, v in equals.items())]


class KeyValueStore(_Store):
    """Live shopping carts: session_id -> cart. Only lookups by the exact key are cheap."""
    family = "key-value"

    def __init__(self, data):
        super().__init__()
        self._kv = {k: v for k, v in data["carts"].items()}

    def get(self, key):
        """The value stored under key, or None. 1 item touched."""
        self.touched += 1
        v = self._kv.get(key)
        return copy.deepcopy(v) if v is not None else None

    def put(self, key, value):
        """Store value under key. 1 item touched."""
        self.touched += 1
        self._kv[key] = copy.deepcopy(value)

    def scan(self):
        """Every (key, value) pair. Touches every key."""
        self.touched += len(self._kv)
        return [(k, copy.deepcopy(v)) for k, v in self._kv.items()]


class WideColumnStore(_Store):
    """Cassandra-style tables. A read names one partition and gets its rows in clustering order.
      orders_by_restaurant_day  partition (restaurant_id, day)  rows newest first: ts, order_id descending
      orders_by_customer        partition customer_id           rows newest first: ts, order_id descending"""
    family = "wide-column"

    def __init__(self, data):
        super().__init__()
        self._t = {"orders_by_restaurant_day": {}, "orders_by_customer": {}}
        for o in data["orders"]:
            row = {"order_id": o["order_id"], "ts": o["ts"], "customer_id": o["customer_id"],
                   "restaurant_id": o["restaurant_id"], "total_cents": o["total_cents"]}
            self._t["orders_by_restaurant_day"].setdefault((o["restaurant_id"], o["day"]), []).append(row)
            self._t["orders_by_customer"].setdefault(o["customer_id"], []).append(row)
        for t in self._t.values():
            for rows in t.values():
                rows.sort(key=lambda r: (r["ts"], r["order_id"]), reverse=True)

    def tables(self):
        return list(self._t)

    def read(self, table, partition, limit=None):
        """Rows of one partition, in clustering order, at most limit of them.
        Items touched = rows returned. table must be one of tables()."""
        if table not in self._t:
            raise KeyError(f"no table {table!r}; the tables are {self.tables()}")
        rows = self._t[table].get(tuple(partition) if isinstance(partition, list) else partition, [])
        rows = rows[:limit] if limit is not None else rows
        self.touched += len(rows)
        return [dict(r) for r in rows]

    def scan(self, table):
        """Every row of a table, partition by partition. Touches every row."""
        out = [dict(r) for rows in self._t[table].values() for r in rows]
        self.touched += len(out)
        return out


class GraphStore(_Store):
    """Nodes are customers (C-....) and restaurants (R-...). Edge types:
      FRIEND      customer -> customer (both directions)
      ORDERED     customer -> restaurant, returned as (restaurant_id, number_of_orders)
      ORDERED_BY  restaurant -> customer, returned as (customer_id, number_of_orders)"""
    family = "graph"

    def __init__(self, data):
        super().__init__()
        self._adj = {"FRIEND": {}, "ORDERED": {}, "ORDERED_BY": {}}
        self._props = {c["customer_id"]: dict(c) for c in data["customers"]}
        self._props.update({r["restaurant_id"]: dict(r) for r in data["restaurants"]})
        for a, b in data["friends"]:
            self._adj["FRIEND"].setdefault(a, []).append(b)
            self._adj["FRIEND"].setdefault(b, []).append(a)
        counts = {}
        for o in data["orders"]:
            k = (o["customer_id"], o["restaurant_id"])
            counts[k] = counts.get(k, 0) + 1
        for (c, r), n in counts.items():
            self._adj["ORDERED"].setdefault(c, []).append((r, n))
            self._adj["ORDERED_BY"].setdefault(r, []).append((c, n))
        for edges in self._adj.values():
            for v in edges.values():
                v.sort()

    def neighbors(self, node, edge):
        """The nodes one edge away. Items touched = edges followed."""
        if edge not in self._adj:
            raise KeyError(f"no edge type {edge!r}; use FRIEND, ORDERED or ORDERED_BY")
        out = self._adj[edge].get(node, [])
        self.touched += len(out)
        return list(out)

    def node(self, node_id):
        """A node's properties. 1 item touched."""
        self.touched += 1
        return dict(self._props.get(node_id, {}))


class VectorStore(_Store):
    """600 dishes, each with a 16-number taste embedding. Similarity is cosine similarity."""
    family = "vector"

    def __init__(self, data):
        super().__init__()
        self._ids = [d["dish_id"] for d in data["dishes"]]
        self._meta = {d["dish_id"]: {k: d[k] for k in ("dish_id", "restaurant_id", "name", "cuisine", "tags", "price_cents")} for d in data["dishes"]}
        self._m = np.array([d["embedding"] for d in data["dishes"]], dtype=float)
        self._norm = np.linalg.norm(self._m, axis=1)
        self._lists = {}
        for i, d in enumerate(data["dishes"]):
            self._lists.setdefault(d["cuisine"], []).append(i)
        self._centroids = {c: self._m[ix].mean(axis=0) for c, ix in self._lists.items()}

    def vector(self, dish_id):
        """The embedding of one dish. 1 item touched."""
        self.touched += 1
        return self._m[self._ids.index(dish_id)].copy()

    def dish(self, dish_id):
        """A dish's name, restaurant, cuisine, tags and price. 1 item touched."""
        self.touched += 1
        return dict(self._meta[dish_id])

    def _rank(self, q, idx, k, exclude):
        q = np.asarray(q, dtype=float)
        s = (self._m[idx] @ q) / (self._norm[idx] * np.linalg.norm(q))
        pairs = sorted(((round(float(v), 9), self._ids[i]) for v, i in zip(s, idx)
                        if self._ids[i] not in exclude), key=lambda t: (-t[0], t[1]))
        return [(d, v) for v, d in pairs[:k]]

    def top_k(self, query, k=5, exclude=()):
        """Exact search: compares the query with every dish. query is a vector or a dish_id.
        Returns [(dish_id, similarity), ...], most similar first. Touches all 600 vectors."""
        if isinstance(query, str):
            query = self.vector(query)
        self.touched += len(self._ids)
        return self._rank(query, list(range(len(self._ids))), k, set(exclude))

    def top_k_ivf(self, query, k=5, nprobe=2, exclude=()):
        """Approximate search with an inverted-file index: the dishes are grouped into 8 clusters.
        Compare the query with the 8 cluster centres, then only with the dishes in the nprobe
        nearest clusters. Fewer items touched, and it can miss a true neighbour."""
        if isinstance(query, str):
            query = self.vector(query)
        q = np.asarray(query, dtype=float)
        self.touched += len(self._centroids)
        near = sorted(self._centroids, key=lambda c: -float(self._centroids[c] @ q) / float(np.linalg.norm(self._centroids[c]) * np.linalg.norm(q)))[:nprobe]
        idx = [i for c in near for i in self._lists[c]]
        self.touched += len(idx)
        return self._rank(q, idx, k, set(exclude))


class SpartanEats:
    def __init__(self, data):
        self.relational = RelationalStore(data)
        self.document = DocumentStore(data)
        self.kv = KeyValueStore(data)
        self.wide = WideColumnStore(data)
        self.graph = GraphStore(data)
        self.vector = VectorStore(data)
        self.stores = [self.relational, self.document, self.kv, self.wide, self.graph, self.vector]

    def reset(self):
        for s in self.stores:
            s.reset()

    def costs(self):
        return {s.family: s.touched for s in self.stores}

    def total(self):
        return sum(s.touched for s in self.stores)


DATA = generate()
db = SpartanEats(DATA)

# --------------------------------------------------------------------------- the eight requests
PATTERNS = {
    "P1": "Reopen the app: show the live cart for session S-02917.",
    "P2": "Order-details screen for order O-11873: the restaurant's name, the order total and every line item (dish name and quantity), in one read.",
    "P3": "Kitchen tablet: the 10 most recent orders at restaurant R-007 on 2026-10-03, newest first.",
    "P4": "\"Your friends' friends love these\": for customer C-0815, the 5 restaurants that the most friends of friends have ordered from, among restaurants C-0815 has never ordered from.",
    "P5": "\"Dishes like this one\": the 5 dishes most similar to dish D-0123 by taste embedding.",
    "P6": "Finance report: revenue by cuisine for restaurants in Sunnyvale, Sep 28 to Oct 4, 2026, highest first.",
    "P7": "Refund order O-04406: move its total from the restaurant's payout balance to the customer's wallet, both changes or neither.",
    "P8": "Order history: every order customer C-1042 placed in the last 30 days, newest first.",
}
PATTERN_IDS = list(PATTERNS)
REQUIRED = ["P2", "P3", "P4", "P5"]
OPTIONAL = ["P6"]
FIELDS = ["session_id", "order_id", "restaurant_id", "customer_id", "dish_id", "day", "ts", "city",
          "cuisine", "embedding", "total_cents", "items", "friend", "balance_cents", "wallet", "payout"]

SHAPES = {
    "P2": 'a dict {"restaurant_name": str, "total_cents": int, "items": [(dish name, qty), ...]} (item order does not matter)',
    "P3": "a list of 10 order_ids, newest first, e.g. ['O-12345', ...]",
    "P4": "a list of 5 restaurant_ids, most friends of friends first, e.g. ['R-012', ...]",
    "P5": "a list of 5 dish_ids, most similar first, e.g. ['D-0042', ...]",
    "P6": "a list of (cuisine, revenue_cents) pairs, highest revenue first",
}
HINTS = {
    "P2": "Dish names come from the line items, quantities from the order, total_cents is the order total.",
    "P3": "Newest first, and when two orders share a timestamp the higher order_id comes first. Exactly 10 order_ids.",
    "P4": "Friends of friends are people who are friends with one of C-0815's friends, excluding C-0815 and C-0815's own friends. Skip restaurants C-0815 has ordered from. Count distinct people per restaurant; ties go to the lower restaurant_id.",
    "P5": "Use the exact search, top_k, for this answer: top_k_ivf is approximate and can miss a neighbour. Do not count D-0123 itself. Ties go to the lower dish_id.",
    "P6": "Orders at restaurants whose city is Sunnyvale, days 2026-09-28 to 2026-10-04 inclusive, sum of total_cents per cuisine, highest first, ties by cuisine A to Z.",
}


def _norm_answer(p, a):
    if p == "P2":
        return [str(a["restaurant_name"]), int(a["total_cents"]), sorted([str(n), int(q)] for n, q in a["items"])]
    if p == "P6":
        return [[str(c), int(v)] for c, v in a]
    return [str(x) for x in a]


def _shape_problem(p, a):
    try:
        n = _norm_answer(p, a)
    except Exception:
        return f"the answer has the wrong shape. Return {SHAPES[p]}."
    want = {"P3": 10, "P4": 5, "P5": 5}.get(p)
    if want is not None and len(n) != want:
        return f"the answer has {len(n)} entries; return {SHAPES[p]}."
    return ""


def _fp(p, a):
    return hashlib.sha256((p + json.dumps(_norm_answer(p, a), separators=(",", ":"))).encode()).hexdigest()[:16]


# --------------------------------------------------------------------------- SQL baselines
def baseline_P2(db):
    rows = db.relational.sql("""
        SELECT r.name, o.total_cents, d.name, i.qty
        FROM orders o JOIN restaurants r ON r.restaurant_id = o.restaurant_id
             JOIN order_items i ON i.order_id = o.order_id
             JOIN dishes d ON d.dish_id = i.dish_id
        WHERE o.order_id = ?""", ("O-11873",))
    return {"restaurant_name": rows[0][0], "total_cents": rows[0][1], "items": [(r[2], r[3]) for r in rows]}


def baseline_P3(db):
    rows = db.relational.sql("""
        SELECT order_id FROM orders WHERE restaurant_id = ? AND day = ?
        ORDER BY ts DESC, order_id DESC LIMIT 10""", ("R-007", "2026-10-03"))
    return [r[0] for r in rows]


def baseline_P4(db):
    rows = db.relational.sql("""
        WITH f AS (SELECT b AS p FROM friends WHERE a = :c),
             fof AS (SELECT DISTINCT f2.b AS p FROM friends f2 JOIN f ON f2.a = f.p
                     WHERE f2.b <> :c AND f2.b NOT IN (SELECT p FROM f)),
             tried AS (SELECT DISTINCT restaurant_id FROM orders WHERE customer_id = :c)
        SELECT o.restaurant_id, COUNT(DISTINCT o.customer_id) AS people
        FROM fof JOIN orders o ON o.customer_id = fof.p
        WHERE o.restaurant_id NOT IN (SELECT restaurant_id FROM tried)
        GROUP BY o.restaurant_id ORDER BY people DESC, o.restaurant_id LIMIT 5""", {"c": "C-0815"})
    return [r[0] for r in rows]


def baseline_P5(db):
    rows = db.relational.sql("SELECT dish_id, embedding FROM dishes")
    ids = [r[0] for r in rows]
    m = np.array([json.loads(r[1]) for r in rows])
    q = m[ids.index("D-0123")]
    s = (m @ q) / (np.linalg.norm(m, axis=1) * np.linalg.norm(q))
    pairs = sorted(((round(float(v), 9), d) for v, d in zip(s, ids) if d != "D-0123"), key=lambda t: (-t[0], t[1]))
    return [d for _, d in pairs[:5]]


def baseline_P6(db):
    rows = db.relational.sql("""
        SELECT r.cuisine, SUM(o.total_cents) AS revenue_cents
        FROM restaurants r JOIN orders o ON o.restaurant_id = r.restaurant_id
        WHERE r.city = ? AND o.day BETWEEN ? AND ?
        GROUP BY r.cuisine ORDER BY revenue_cents DESC, r.cuisine""", ("Sunnyvale", "2026-09-28", "2026-10-04"))
    return [(r[0], r[1]) for r in rows]


BASELINES = {"P2": baseline_P2, "P3": baseline_P3, "P4": baseline_P4, "P5": baseline_P5, "P6": baseline_P6}
FINGERPRINTS = {"P2": "b3eb906cc8e1cb3e", "P3": "4610180f2737c121", "P4": "a14dd20843c35ee8", "P5": "f5ecf958dffa6aa3", "P6": "c52d0c0ef5a1658b"}
RESULTS = {}


def check(p, fn, store, note=""):
    """Run the SQL baseline and your query, compare your answer with the right one, print both costs."""
    db.reset()
    base = BASELINES[p](db)
    base_cost = db.total()
    same_data = _fp(p, base) == FINGERPRINTS[p]
    db.reset()
    status, why, ans = "NOT YET", "", None
    try:
        ans = fn(db)
        if ans is None:
            why = "the query returned None. End it with return and the answer."
    except NotImplementedError:
        why = "the query is not written yet (it still raises NotImplementedError)."
    except Exception as e:
        why = f"the query raised {type(e).__name__}: {e}"
    costs = db.costs()
    if not why:
        why = _shape_problem(p, ans)
        if not why:
            right = (_fp(p, ans) == FINGERPRINTS[p]) if same_data else (_norm_answer(p, ans) == _norm_answer(p, base))
            if right:
                status = "PASS"
            else:
                why = "the answer does not match. " + HINTS[p]
    store_s = store.strip().lower() if isinstance(store, str) else ""
    mismatch = store_s in FAMILIES and status == "PASS" and costs.get(store_s, 0) == 0
    RESULTS[p] = {"status": status, "why": why, "store": store_s, "cost": sum(costs.values()),
                  "costs": {k: v for k, v in costs.items() if v}, "base_cost": base_cost,
                  "note": note if isinstance(note, str) else "", "mismatch": mismatch}
    used = ", ".join(f"{k} {v:,}" for k, v in RESULTS[p]["costs"].items()) or "nothing"
    print(f"{p}: {status}" + ("" if status == "PASS" else f". {why}"))
    print(f"   your query:   {RESULTS[p]['cost']:>7,} items touched ({used})")
    print(f"   SQL baseline: {base_cost:>7,} items touched (relational)")
    if store_s not in FAMILIES:
        print(f"   STORE_{p} must be one of {', '.join(FAMILIES)}.")
    elif mismatch:
        print(f"   STORE_{p} says {store_s}, but your query did not touch the {store_s} store.")
    if not same_data:
        print("   (This runtime generated different data from the instructor's; the answer was compared with the baseline instead.)")


print(f"SpartanEats: {len(DATA['customers']):,} customers, {len(DATA['restaurants'])} restaurants, "
      f"{len(DATA['dishes'])} dishes, {len(DATA['orders']):,} orders, {len(DATA['friends']):,} friendships, "
      f"{len(DATA['carts']):,} live carts. Six stores loaded.")

SpartanEats: 2,000 customers, 150 restaurants, 600 dishes, 20,000 orders, 4,000 friendships, 3,000 live carts. Six stores loaded.


## Store guide

Everything you can call. Each call adds to its store's count of items touched. Use only these
methods: reading a store's private attributes (the ones starting with `_`) skips the meter, and the
comparison then means nothing.

| Store | Family | Calls | What one call touches |
|---|---|---|---|
| `db.relational` | relational | `sql(query, params)` returns a list of tuples; `explain(query, params)` prints the plan; `schema()` prints the tables | every row SQLite reads, including each index lookup |
| `db.document` | document | `get(order_id)`; `find(field=value, ...)` | `get`: 1 document. `find`: all 20,000 documents (no secondary index) |
| `db.kv` | key-value | `get(key)`, `put(key, value)`, `scan()` | `get`/`put`: 1 key. `scan`: every key |
| `db.wide` | wide-column | `read(table, partition, limit=None)`, `scan(table)`, `tables()` | the rows returned from that one partition |
| `db.graph` | graph | `neighbors(node, edge)`, `node(node_id)` | the edges followed |
| `db.vector` | vector | `top_k(query, k, exclude)`, `top_k_ivf(query, k, nprobe, exclude)`, `vector(dish_id)`, `dish(dish_id)` | the vectors compared (all 600 for `top_k`) |

**What is in each store**

- **relational**: tables `customers`, `restaurants`, `dishes` (with `embedding` as JSON text), `orders`, `order_items`, `friends` (both directions), `carts`, `wallets`, `payouts`. Indexes on the primary keys and on `orders(customer_id, ts)`, `orders(restaurant_id, day)`, `orders(day)`, `order_items(order_id)`, `friends(a, b)`, `restaurants(city)`, `dishes(restaurant_id)`.
- **document**: one document per order, keyed by `order_id`, with `customer_id`, `restaurant_id`, `restaurant_name`, `cuisine`, `city`, `ts`, `day`, `total_cents` and `items` (each with `dish_id`, `name`, `qty`, `price_cents`).
- **key-value**: live carts, `session_id` (like `S-00042`) to `{customer_id, restaurant_id, items, updated}`.
- **wide-column**: `orders_by_restaurant_day`, partition `(restaurant_id, day)`, and `orders_by_customer`, partition `customer_id`. Rows in both are newest first.
- **graph**: customer and restaurant nodes. Edges `FRIEND` (customer to customer), `ORDERED` (customer to restaurant, with a count) and `ORDERED_BY` (restaurant to customer, with a count).
- **vector**: 600 dishes with a 16-number taste embedding; cosine similarity.

Field names to use for access keys: `session_id`, `order_id`, `restaurant_id`, `customer_id`,
`dish_id`, `day`, `ts`, `city`, `cuisine`, `embedding`, `total_cents`, `balance_cents`.

Run the next cell to see the meter work.

In [3]:
# Two lookups for the same cart, and what each one touched.
db.reset()
cart = db.kv.get("S-02917")
print("key-value get:", cart, "| items touched:", db.kv.touched)

db.reset()
rows = db.relational.sql("SELECT * FROM carts WHERE session_id = ?", ("S-02917",))
print("relational, by primary key:", len(rows), "row | items touched:", db.relational.touched)

db.reset()
rows = db.relational.sql("SELECT * FROM carts WHERE updated LIKE ?", ("2026-10-14 21:%",))
print("relational, no index on updated:", len(rows), "rows | items touched:", db.relational.touched)
db.relational.explain("SELECT * FROM carts WHERE updated LIKE ?", ("2026-10-14 21:%",))

key-value get: {'customer_id': 'C-0315', 'restaurant_id': 'R-015', 'items': [{'dish_id': 'D-0059', 'qty': 1}, {'dish_id': 'D-0060', 'qty': 2}], 'updated': '2026-10-14 13:26'} | items touched: 1
relational, by primary key: 1 row | items touched: 1
relational, no index on updated: 223 rows | items touched: 3000
   SCAN _carts


## Round 1. Match (10 minutes)

The eight requests SpartanEats has to serve:

| | Request |
|---|---|
| **P1** | Reopen the app: show the live cart for session S-02917. |
| **P2** | Order-details screen for order O-11873: the restaurant's name, the order total and every line item (dish name and quantity), in one read. |
| **P3** | Kitchen tablet: the 10 most recent orders at restaurant R-007 on 2026-10-03, newest first. |
| **P4** | "Your friends' friends love these": for customer C-0815, the 5 restaurants that the most friends of friends have ordered from, among restaurants C-0815 has never ordered from. |
| **P5** | "Dishes like this one": the 5 dishes most similar to dish D-0123 by taste embedding. |
| **P6** | Finance report: revenue by cuisine for restaurants in Sunnyvale, Sep 28 to Oct 4, 2026, highest first. |
| **P7** | Refund order O-04406: move its total from the restaurant's payout balance to the customer's wallet, both changes or neither. |
| **P8** | Order history: every order customer C-1042 placed in the last 30 days, newest first. |

For each request write:

- **family**: the store family that should answer it: `relational`, `document`, `key-value`, `wide-column`, `graph` or `vector`. Name the family, not a product.
- **key**: the field or fields the store looks the data up by, using the field names from the store guide (for example `order_id`, or `restaurant_id, day`).
- **reason**: one sentence that says why this family fits this request. Name the lookup it makes cheap or the guarantee it gives. "Faster" or "scalable" alone is not a reason.

Two of the eight have an obvious-looking answer that is wrong. Read P7 and P8 twice.

In [4]:
MATCH = {
    "P1": {"family": "key-value", "key": "session_id",
           "reason": "The live cart is read and written whole by its exact session_id, so a key-value get touches one key with no schema, joins or index to maintain."},
    "P2": {"family": "document", "key": "order_id",
           "reason": "The order document already embeds the restaurant name, the total and every line item, so one get by order_id renders the whole screen with no joins."},
    "P3": {"family": "wide-column", "key": "restaurant_id, day (clustered by ts, order_id descending)",
           "reason": "The orders_by_restaurant_day partition keyed by (restaurant_id, day) stores rows newest first, so reading the first 10 rows of one partition touches exactly 10 rows and needs no sort."},
    "P4": {"family": "graph", "key": "customer_id (FRIEND edges, then ORDERED edges)",
           "reason": "Friends of friends is a two-hop traversal, so a graph follows only C-0815's FRIEND edges and those people's ORDERED edges instead of self-joining the whole friends table and then the orders table."},
    "P5": {"family": "vector", "key": "dish_id, then its embedding",
           "reason": "Similar dishes is a nearest-neighbour search by cosine similarity on the 16-number taste embedding, which a vector store does natively (with an optional IVF index) rather than by equality on any key."},
    "P6": {"family": "relational", "key": "city, day (restaurants by city joined to orders by restaurant_id, day)",
           "reason": "It is an ad-hoc report that joins restaurants filtered by city to orders filtered by a day range with GROUP BY and SUM, which SQL answers through the restaurants(city) and orders(restaurant_id, day) indexes instead of scanning every order."},
    "P7": {"family": "relational", "key": "order_id, then customer_id (wallets) and restaurant_id (payouts)",
           "reason": "The refund changes two balances in two tables, and a relational store gives an ACID transaction so the payout debit and the wallet credit commit together or not at all; two separate key-value puts could leave the money half-moved."},
    "P8": {"family": "wide-column", "key": "customer_id, ts",
           "reason": "Order history is looked up by customer_id, not order_id, and orders_by_customer partitions by customer_id with rows newest first, so one partition read touches only that customer's orders, while the document store's find would read all 20,000."},
}
for p, m in MATCH.items():
    print(f"{p}: {m['family']:<12} key: {m['key']}")

P1: key-value    key: session_id
P2: document     key: order_id
P3: wide-column  key: restaurant_id, day (clustered by ts, order_id descending)
P4: graph        key: customer_id (FRIEND edges, then ORDERED edges)
P5: vector       key: dish_id, then its embedding
P6: relational   key: city, day (restaurants by city joined to orders by restaurant_id, day)
P7: relational   key: order_id, then customer_id (wallets) and restaurant_id (payouts)
P8: wide-column  key: customer_id, ts


## Round 2. Date (20 minutes)

Now test four of your matches: **P2, P3, P4 and P5.** For each one:

1. Set `STORE_Px` to the family you query. Start with your round-1 choice. If it turns out to be the
   wrong store, change it, and say so in your round-3 sentence.
2. Write the body of `answer_Px(db)` using the calls in the store guide, and `return` the answer in
   the shape the cell asks for. Choosing `relational` is allowed: then write SQL with `db.relational.sql`.
3. Run the cell. It runs a SQL baseline, runs your query, checks your answer against the right
   answer, and prints both costs in items touched. `PASS` means your answer is right.

If a query will not work, set `NOTE_Px = "BROKE: what went wrong"` and move on.
**P6 is optional.** Do it if you finish early.

In [5]:
# P2. Order-details screen for order O-11873: the restaurant's name, the order total and every line item (dish name and quantity), in one read.
STORE_P2 = "document"

def answer_P2(db):
    """Return "restaurant_name": str, "total_cents": int, "items": [(dish name, qty), ...]."""
    d = db.document.get("O-11873")          # one read: the document embeds the restaurant name and line items
    return {"restaurant_name": d["restaurant_name"], "total_cents": d["total_cents"],
            "items": [(i["name"], i["qty"]) for i in d["items"]]}

NOTE_P2 = ""   # Only if you cannot get it working: "BROKE: what went wrong"
check("P2", answer_P2, STORE_P2, NOTE_P2)

P2: PASS
   your query:         1 items touched (document 1)
   SQL baseline:       8 items touched (relational)


In [6]:
# P3. Kitchen tablet: the 10 most recent orders at restaurant R-007 on 2026-10-03, newest first.
STORE_P3 = "wide-column"

def answer_P3(db):
    """Return a list of 10 order_ids, newest first (equal timestamps: higher order_id first)."""
    # One partition (restaurant_id, day); rows are already clustered by ts, order_id descending.
    rows = db.wide.read("orders_by_restaurant_day", ("R-007", "2026-10-03"), limit=10)
    return [r["order_id"] for r in rows]

NOTE_P3 = ""   # Only if you cannot get it working: "BROKE: what went wrong"
check("P3", answer_P3, STORE_P3, NOTE_P3)

P3: PASS
   your query:        10 items touched (wide-column 10)
   SQL baseline:      91 items touched (relational)


In [7]:
# P4. "Your friends' friends love these": for customer C-0815, the 5 restaurants that the most friends of friends have ordered from, among restaurants C-0815 has never ordered from.
STORE_P4 = "graph"

def answer_P4(db):
    """Return a list of 5 restaurant_ids, most friends of friends first (ties: lower restaurant_id)."""
    me = "C-0815"
    friends = set(db.graph.neighbors(me, "FRIEND"))                 # hop 1
    fof = set()
    for f in friends:                                               # hop 2
        fof.update(db.graph.neighbors(f, "FRIEND"))
    fof -= friends | {me}                                           # not me, not my own friends
    tried = {r for r, _ in db.graph.neighbors(me, "ORDERED")}       # restaurants I have ordered from
    people = {}                                                     # restaurant -> distinct friends of friends
    for p in fof:
        for r, _ in db.graph.neighbors(p, "ORDERED"):              # one edge per (person, restaurant)
            if r not in tried:
                people[r] = people.get(r, 0) + 1
    ranked = sorted(people.items(), key=lambda t: (-t[1], t[0]))
    return [r for r, _ in ranked[:5]]

NOTE_P4 = ""   # Only if you cannot get it working: "BROKE: what went wrong"
check("P4", answer_P4, STORE_P4, NOTE_P4)

P4: PASS
   your query:       235 items touched (graph 235)
   SQL baseline:     363 items touched (relational)


In [8]:
# P5. "Dishes like this one": the 5 dishes most similar to dish D-0123 by taste embedding.
STORE_P5 = "vector"

def answer_P5(db):
    """Return a list of 5 dish_ids, most similar first, not D-0123 itself."""
    # Exact search (top_k), as the hint asks; exclude the dish itself.
    return [d for d, _ in db.vector.top_k("D-0123", k=5, exclude=["D-0123"])]

NOTE_P5 = ""   # Only if you cannot get it working: "BROKE: what went wrong"
check("P5", answer_P5, STORE_P5, NOTE_P5)

P5: PASS
   your query:       601 items touched (vector 601)
   SQL baseline:     600 items touched (relational)


### Optional: P6, if you finish early

In [9]:
# P6. Finance report: revenue by cuisine for restaurants in Sunnyvale, Sep 28 to Oct 4, 2026, highest first.
STORE_P6 = "relational"

def answer_P6(db):
    """Return a list of (cuisine, revenue_cents) pairs, highest first (ties: cuisine A to Z)."""
    rows = db.relational.sql("""
        SELECT r.cuisine, SUM(o.total_cents) AS revenue_cents
        FROM restaurants r JOIN orders o ON o.restaurant_id = r.restaurant_id
        WHERE r.city = ? AND o.day BETWEEN ? AND ?
        GROUP BY r.cuisine
        ORDER BY revenue_cents DESC, r.cuisine""", ("Sunnyvale", "2026-09-28", "2026-10-04"))
    return [(cuisine, revenue) for cuisine, revenue in rows]

NOTE_P6 = ""   # Only if you cannot get it working: "BROKE: what went wrong"
check("P6", answer_P6, STORE_P6, NOTE_P6)

P6: PASS
   your query:       793 items touched (relational 793)
   SQL baseline:     793 items touched (relational)


## Round 3. Two-store budget (10 minutes)

The CTO has budget for **two** stores at launch, not six. Pick the two families SpartanEats keeps,
and assign every request P1 to P8 to one of them.

For every request you **moved** away from its round-1 family, write its cost in `moved`: the items
touched from round 2 if you measured it, or your estimate of items touched (a number, with how you
got it). Leave the other entries empty.

Then one sentence in `lose`: what SpartanEats gives up with two stores, and which request feels it most.

In [10]:
BUDGET = {
    "stores": ["relational", "wide-column"],
    "assign": {"P1": "relational", "P2": "relational", "P3": "wide-column", "P4": "relational",
               "P5": "relational", "P6": "relational", "P7": "relational", "P8": "wide-column"},
    # Only for requests whose family changed from round 1, e.g. "P5": "about 600: top_k compares every dish".
    "moved": {
        "P1": "1 row, measured: the store-guide cell's SELECT on carts by primary key session_id touched 1 row, the same as the key-value get",
        "P2": "8 rows, measured: the round-2 SQL baseline joins orders, restaurants, order_items and dishes (1 + 1 + 3 + 3 rows) instead of 1 document",
        "P3": "",
        "P4": "363 rows, measured: the round-2 SQL baseline (friends self-join, then orders per friend of a friend) instead of 235 graph edges",
        "P5": "600 rows, measured: the round-2 SQL baseline reads every dish embedding and ranks in Python, about the same as exact top_k (601), but with no IVF index to cut it",
        "P6": "",
        "P7": "",
        "P8": "",
    },
    "lose": "With only relational and wide-column, SpartanEats loses the one-read order document (P2 becomes a 4-table join, 8 rows instead of 1), the vector index (P5 must score all 600 embeddings with no approximate IVF shortcut) and the graph traversal, and P4 feels it most: friends of friends becomes a self-join plus per-person order lookups (363 rows instead of 235 edges) whose cost grows with every extra hop and every new friendship.",
}
print("Two stores:", BUDGET["stores"])

Two stores: ['relational', 'wide-column']


## Check your work

Run the validation cell. It checks that every part is complete. It does not grade your choices;
it lists anything missing or too vague to read. Fix each item it lists and run it again until it
prints `READY`.

In [11]:
def _filled(v):
    return isinstance(v, str) and v.strip() != "" and "REPLACE" not in v

def _broke(v):
    return isinstance(v, str) and v.strip().upper().startswith("BROKE:")

PRODUCTS = {"mongodb": "document", "mongo": "document", "couchdb": "document", "firestore": "document",
            "redis": "key-value", "memcached": "key-value", "dynamodb": "key-value", "valkey": "key-value",
            "cassandra": "wide-column", "scylladb": "wide-column", "hbase": "wide-column", "bigtable": "wide-column",
            "neo4j": "graph", "neptune": "graph", "pgvector": "vector", "pinecone": "vector", "milvus": "vector",
            "weaviate": "vector", "postgres": "relational", "postgresql": "relational", "mysql": "relational",
            "sqlite": "relational", "oracle": "relational", "sql": "relational", "nosql": None}
STOP = set("a an the it its it's is are be to of for and or in on at by with this that than more most much very so because since as we our us they them one ones other others which when what can will would just also only all any each from into".split())
BUZZ = {"faster", "fast", "quick", "quicker", "speed", "scalable", "scales", "scale", "scaling", "better",
        "best", "efficient", "efficiency", "performance", "performant", "flexible", "modern", "easy", "easier",
        "simple", "simpler", "good", "great", "nosql", "optimized", "optimal", "suited", "suitable", "perfect", "ideal"}

def _family_problems(v, where):
    if not _filled(v):
        return [f"{where}: still says REPLACE."]
    s = v.strip().lower()
    if s in FAMILIES:
        return []
    if s in PRODUCTS:
        fam = PRODUCTS[s]
        if fam is None:
            return [f"{where}: 'NoSQL' is four families, not one. Pick document, key-value, wide-column or graph."]
        return [f"{where}: {v.strip()} is a product. It is a {fam} store; write the family, {fam}."]
    return [f"{where}: '{v.strip()}' is not a family. Use one of: {', '.join(FAMILIES)}."]

def _key_problems(v, where):
    if not _filled(v):
        return [f"{where}: still says REPLACE."]
    low = v.lower()
    if not any(f in low for f in FIELDS):
        return [f"{where}: '{v.strip()}' does not name a field. Use the field names from the store guide, for example order_id or restaurant_id, day."]
    return []

def _reason_problems(v, where, minwords=8):
    if not _filled(v):
        return [f"{where}: still says REPLACE."]
    low = v.strip().lower()
    if low.startswith("it depends"):
        return [f"{where}: 'it depends' is where the reason starts, not the reason. Say what it depends on and which way this request goes."]
    words = re.findall(r"[a-z][a-z_\-']*", low)
    content = [w for w in words if w not in STOP]
    if content and all(w in BUZZ or w in FAMILIES or w in ("store", "database", "db", "data") for w in content):
        return [f"{where}: '{v.strip()[:40]}' says it is better, not why. Name the lookup it makes cheap (by which key) or the guarantee it gives."]
    if len(words) < minwords:
        return [f"{where}: write a full sentence (at least {minwords} words) that names the lookup or the guarantee."]
    return []

def _group_problems():
    out = []
    if not isinstance(GROUP, list) or not (1 <= len(GROUP) <= 8):
        out.append("GROUP: list every member of your team, one string per person.")
    else:
        for i, g in enumerate(GROUP, 1):
            if not _filled(g):
                out.append(f"GROUP: member {i} still says REPLACE.")
        names = [g.strip().lower() for g in GROUP if isinstance(g, str)]
        if len(set(names)) != len(names):
            out.append("GROUP: the same name appears twice.")
    if not _filled(TEAM_NAME):
        out.append("TEAM_NAME: still says REPLACE.")
    return out

def _problems():
    out = _group_problems()
    for p in PATTERN_IDS:
        m = MATCH.get(p)
        if not isinstance(m, dict):
            out.append(f"MATCH {p}: missing. Copy its line back from a fresh copy of the notebook.")
            continue
        out += _family_problems(m.get("family"), f"Round 1 {p} family")
        out += _key_problems(m.get("key"), f"Round 1 {p} key")
        out += _reason_problems(m.get("reason"), f"Round 1 {p} reason")
    for p in REQUIRED:
        r = RESULTS.get(p)
        if r is None:
            out.append(f"Round 2 {p}: its cell has not run. Run it; it prints PASS or NOT YET.")
            continue
        if r["status"] != "PASS":
            if _broke(r["note"]) and len(r["note"].split()) >= 4:
                continue
            if _broke(r["note"]):
                out.append(f"Round 2 {p}: BROKE: is fine, but say what went wrong in a few words.")
            else:
                out.append(f"Round 2 {p}: NOT YET, {r['why']} Fix the query, or set NOTE_{p} = \"BROKE: what went wrong\" and run the cell again.")
            continue
        if r["store"] not in FAMILIES:
            out.append(f"Round 2 {p}: STORE_{p} must be one of {', '.join(FAMILIES)}.")
        elif r["mismatch"]:
            out.append(f"Round 2 {p}: STORE_{p} says {r['store']}, but the query touched only {', '.join(r['costs'])}. Set STORE_{p} to the store you queried.")
    st = BUDGET.get("stores")
    if not (isinstance(st, list) and len(st) == 2):
        out.append("Round 3 stores: list exactly two families.")
        st = []
    else:
        for i, s in enumerate(st, 1):
            out += _family_problems(s, f"Round 3 store {i}")
        st = [s.strip().lower() for s in st if isinstance(s, str)]
        if len(st) == 2 and st[0] == st[1] and st[0] in FAMILIES:
            out.append("Round 3 stores: the two stores must be two different families.")
    assign = BUDGET.get("assign", {})
    moved = BUDGET.get("moved", {})
    for p in PATTERN_IDS:
        a = assign.get(p)
        if not _filled(a):
            out.append(f"Round 3 assign {p}: still says REPLACE.")
            continue
        a = a.strip().lower()
        if len(st) == 2 and a not in st:
            out.append(f"Round 3 assign {p}: '{a}' is not one of your two stores ({st[0]}, {st[1]}).")
            continue
        r1 = MATCH.get(p, {}).get("family", "")
        r1 = r1.strip().lower() if isinstance(r1, str) else ""
        if r1 in FAMILIES and a != r1:
            mv = moved.get(p, "")
            if _broke(mv):
                continue
            if not _filled(mv) or not re.search(r"\d", mv):
                out.append(f"Round 3 moved {p}: {p} moved from {r1} to {a}. Write its items touched from round 2, or an estimate with a number and how you got it.")
    out += _reason_problems(BUDGET.get("lose"), "Round 3 lose")
    return out

problems = _problems()
if problems:
    print("NOT READY. Still to do:")
    for x in problems:
        print("  -", x)
else:
    print("READY. Now: Runtime > Restart and run all, check every cell has output, then")
    print("File > Download > Download .ipynb, and one member uploads it to In-class exercise: Data Model.")

READY. Now: Runtime > Restart and run all, check every cell has output, then
File > Download > Download .ipynb, and one member uploads it to In-class exercise: Data Model.


In [12]:
# The graded view: this is what gets read first.
print("Team:", TEAM_NAME, "|", ", ".join(GROUP))
print("\nRound 1")
for p in PATTERN_IDS:
    m = MATCH.get(p, {})
    print(f"  {p}  {str(m.get('family')):<12} key: {m.get('key')}")
    print(f"      {m.get('reason')}")
print("\nRound 2                  status    you    SQL baseline")
done, mine, base = 0, 0, 0
for p in REQUIRED + OPTIONAL:
    r = RESULTS.get(p)
    if r is None:
        print(f"  {p}  (not run)")
        continue
    tag = r["status"] if r["status"] == "PASS" or not _broke(r["note"]) else "BROKE"
    print(f"  {p}  {r['store']:<12}  {tag:<8} {r['cost']:>6,}  {r['base_cost']:>8,}")
    if r["status"] == "PASS" and p in REQUIRED:
        done, mine, base = done + 1, mine + r["cost"], base + r["base_cost"]
    if tag == "BROKE":
        print(f"      {r['note']}")
opt = RESULTS.get("P6", {}).get("status") == "PASS"
print("\nRound 3  stores:", BUDGET.get("stores"))
for p in PATTERN_IDS:
    mv = BUDGET.get("moved", {}).get(p, "")
    print(f"  {p} -> {BUDGET.get('assign', {}).get(p)}" + (f"   moved: {mv}" if mv else ""))
print("  lose:", BUDGET.get("lose"))
print(f"\nCost score: {done} of 4 required answers right" + (", plus P6" if opt else "") +
      f". Items touched on the right ones: yours {mine:,}, SQL baseline {base:,}.")

Team: Capybara Labs | Arda Korkmaz, Mike Chau, Meron Tesfandrias, Mann Malviya

Round 1
  P1  key-value    key: session_id
      The live cart is read and written whole by its exact session_id, so a key-value get touches one key with no schema, joins or index to maintain.
  P2  document     key: order_id
      The order document already embeds the restaurant name, the total and every line item, so one get by order_id renders the whole screen with no joins.
  P3  wide-column  key: restaurant_id, day (clustered by ts, order_id descending)
      The orders_by_restaurant_day partition keyed by (restaurant_id, day) stores rows newest first, so reading the first 10 rows of one partition touches exactly 10 rows and needs no sort.
  P4  graph        key: customer_id (FRIEND edges, then ORDERED edges)
      Friends of friends is a two-hop traversal, so a graph follows only C-0815's FRIEND edges and those people's ORDERED edges instead of self-joining the whole friends table and then the orders 